In [1]:
"""
Converts FLIR-aligned (flir_align/) to YOLO format for ultralytics.

Confirmed dataset structure (from actual inspection):
    flir_align/
        coco_annotations/{train_new,test_new}.json   <- use the _new files (dog already stripped)
        visible/{train,test}/*.jpeg
        thermal/{train,test}/*.jpeg   (not used yet -- Phase 1 is RGB-only)

Annotation format confirmed:
    - image_id is a string like "FLIR_00258", matching file_name minus extension
    - bbox is standard COCO [x_min, y_min, width, height]
    - category_id: 0=car, 1=person, 2=bicycle (dog already removed in *_new.json)

USAGE:
    Edit RAW_ROOT below, then: python prepare_flir_aligned.py
"""

import json
import shutil
from pathlib import Path

# ---- PATHS: dataset lives in Datasets/, scripts live in Thermal_Fusion_Paper/ ----
RAW_ROOT = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_align")
OUT_ROOT = Path(r"C:\Users\LENOVO\Documents\Datasets\flir_aligned_yolo")
# -----------------------------------------------------------------------------

SPLITS = {
    "train": RAW_ROOT / "coco_annotations" / "train_new.json",
    "val": RAW_ROOT / "coco_annotations" / "test_new.json",  # FLIR-aligned's "test" is our val
}
VISIBLE_DIR = {
    "train": RAW_ROOT / "visible" / "train",
    "val": RAW_ROOT / "visible" / "test",
}

# category_id -> YOLO class index. Confirmed order: 0=car, 1=person, 2=bicycle
CLASS_NAMES = ["car", "person", "bicycle"]


def convert_split(split_name: str):
    json_path = SPLITS[split_name]
    img_dir = VISIBLE_DIR[split_name]

    with open(json_path) as f:
        coco = json.load(f)

    images_by_id = {img["id"]: img for img in coco["images"]}
    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    out_img_dir = OUT_ROOT / "images" / split_name
    out_lbl_dir = OUT_ROOT / "labels" / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    n_written, n_missing = 0, 0
    for img_id, img_info in images_by_id.items():
        file_name = img_info["file_name"]  # e.g. "FLIR_00258.jpeg"
        src_path = img_dir / file_name
        if not src_path.exists():
            n_missing += 1
            continue

        w, h = img_info["width"], img_info["height"]
        anns = anns_by_image.get(img_id, [])

        lines = []
        for ann in anns:
            cls_idx = ann["category_id"]  # already 0/1/2, matches CLASS_NAMES order
            x, y, bw, bh = ann["bbox"]
            xc = (x + bw / 2) / w
            yc = (y + bh / 2) / h
            nbw = bw / w
            nbh = bh / h
            lines.append(f"{cls_idx} {xc:.6f} {yc:.6f} {nbw:.6f} {nbh:.6f}")

        stem = img_id  # e.g. "FLIR_00258"
        shutil.copy(src_path, out_img_dir / f"{stem}.jpeg")
        with open(out_lbl_dir / f"{stem}.txt", "w") as f:
            f.write("\n".join(lines))
        n_written += 1

    print(f"[{split_name}] wrote {n_written} pairs, {n_missing} images missing on disk")


def main():
    for split in SPLITS:
        print(f"json: {SPLITS[split]} exists={SPLITS[split].exists()}")
        print(f"img dir: {VISIBLE_DIR[split]} exists={VISIBLE_DIR[split].exists()}")
    print()
    convert_split("train")
    convert_split("val")


if __name__ == "__main__":
    main()

json: C:\Users\LENOVO\Documents\Datasets\flir_align\coco_annotations\train_new.json exists=True
img dir: C:\Users\LENOVO\Documents\Datasets\flir_align\visible\train exists=True
json: C:\Users\LENOVO\Documents\Datasets\flir_align\coco_annotations\test_new.json exists=True
img dir: C:\Users\LENOVO\Documents\Datasets\flir_align\visible\test exists=True

[train] wrote 4129 pairs, 0 images missing on disk
[val] wrote 1013 pairs, 0 images missing on disk
